In [ ]:
import sys, os, glob, shutil, traceback
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
name = torch.cuda.get_device_name(0); print("GPU:", name, flush=True)
if "T4" not in name: raise SystemExit(f"нужна T4, дали {name}")
mods = glob.glob("/kaggle/input/**/train_ce_large.py", recursive=True)
os.makedirs("/kaggle/working/src", exist_ok=True)
for p in glob.glob(os.path.dirname(mods[0]) + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
pack = os.path.dirname(glob.glob("/kaggle/input/**/item_texts.parquet", recursive=True)[0])
os.makedirs("/kaggle/working/pack", exist_ok=True)
for p in glob.glob(pack + "/*"):
    d = "/kaggle/working/pack/" + os.path.basename(p)
    if not os.path.exists(d): os.symlink(p, d)
stage1 = os.path.dirname([p for p in glob.glob("/kaggle/input/**/model.safetensors", recursive=True) if "pack" not in p][0])
print("чекпоинт:", stage1, flush=True)
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.train_ce_large import main
for tag, args in [("relax_lr2e5", ["--human-learning-rate","2e-5"]),
                  ("relax_lr3e5", ["--human-learning-rate","3e-5"])]:
    print("=" * 60 + f"\nКОНФИГУРАЦИЯ {tag}\n" + "=" * 60, flush=True)
    sys.argv = ["train_ce_large","--prepacked","/kaggle/working/pack","--holdout-fold","0",
                "--base-model","DeepPavlov/rubert-base-cased","--resume-from",stage1,
                "--human-relaxed","--human-epochs","2","--batch-size","256","--max-length","256",
                "--output", f"/kaggle/working/{tag}"] + args
    try: main()
    except Exception:
        traceback.print_exc(); print(f"{tag} УПАЛА, идём дальше", flush=True)
